# **The notebook executes the complete ASIC design flow sequentially:**

1.   Synthesis - Converts high-level Verilog to gate-level netlist using standard cells
2.   Floorplanning - Determines chip dimensions and creates the cell placement grid
3.   Tap/Endcap Cell Insertion - Places boundary cells and tap cells for substrate connections
4.   I/O Placement - Places pins at the design boundaries
5.   Power Distribution Network (PDN) Generation - Creates power straps with custom settings for this small design
6.   Global Placement - Initial fuzzy placement of cells
7. Detailed Placement - Finalizes and "legalizes" the cell placement
8. Clock Tree Synthesis (CTS) - Creates balanced clock distribution network
9. Global Routing - Plans wire paths between components
10. Detailed Routing - Creates actual metal connections following the routing guides
11. Fill Insertion - Adds decoupling capacitors and fill cells in empty spaces
12. Parasitics Extraction (RCX) - Computes parasitic effects for timing analysis
13. Static Timing Analysis (STA) - Verifies timing constraints are met
14. Stream-out - Converts design to GDSII format for fabrication
15. Design Rule Checking (DRC) - Verifies the design meets fabrication rules
16. SPICE Extraction - Extracts SPICE netlist from the layout
17. Layout vs. Schematic (LVS) - Validates physical layout matches the logical design


In [ ]:
# @title Setup Nix {display-mode: "form"}
# @markdown <img src="https://raw.githubusercontent.com/NixOS/nixos-artwork/master/logo/nix-snowflake.svg" width="32"/>
# @markdown
# @markdown Nix is a package manager with an emphasis on reproducible builds,
# @markdown and it is the primary method for installing OpenLane 2.
# @markdown
# @markdown This step installs the Nix package manager and enables the
# @markdown experimental "flakes" feature.
import os
import sys
import shutil

os.environ["LOCALE_ARCHIVE"] = "/usr/lib/locale/locale-archive"

if "google.colab" in sys.modules:
    if shutil.which("nix-env") is None:
        !curl -L https://nixos.org/nix/install | bash -s -- --daemon --yes
        !echo "extra-experimental-features = nix-command flakes" >> /etc/nix/nix.conf
        !killall nix-daemon
else:
    if shutil.which("nix-env") is None:
        raise RuntimeError("Nix is not installed!")

os.environ["PATH"] = f"/nix/var/nix/profiles/default/bin/:{os.getenv('PATH')}"

In [ ]:
# @title Get OpenLane {display-mode: "form"}

# @markdown This will install OpenLane's tool dependencies using Nix,
# @markdown and OpenLane itself using PIP.
# @markdown
# @markdown Note that `python3-tk` may need to be installed using your OS's
# @markdown package manager.
import os
import subprocess
import IPython

openlane_version = "version-2.1"  # @param {key:"OpenLane Version", type:"string"}

if openlane_version == "latest":
    openlane_version = "main"

pdk_root = "~/.volare"  # @param {key:"PDK Root", type:"string"}

pdk_root = os.path.expanduser(pdk_root)

pdk = "sky130"  # @param {key:"PDK (without the variant)", type:"string"}

openlane_ipynb_path = os.path.join(os.getcwd(), "openlane_ipynb")

display(IPython.display.HTML("<h3>Downloading OpenLane…</a>"))


TESTING_LOCALLY = False
!rm -rf {openlane_ipynb_path}
!mkdir -p {openlane_ipynb_path}
if TESTING_LOCALLY:
    !ln -s {os.getcwd()} {openlane_ipynb_path}
else:
    !curl -L "https://github.com/efabless/openlane2/tarball/{openlane_version}" | tar -xzC {openlane_ipynb_path} --strip-components 1

try:
    import tkinter
except ImportError:
    if "google.colab" in sys.modules:
        !sudo apt-get install python-tk

try:
    import tkinter
except ImportError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to import the <code>tkinter</code> library for Python, which is required to load PDK configuration values. Make sure <code>python3-tk</code> or equivalent is installed on your system.</a>'
        )
    )
    raise e from None


display(IPython.display.HTML("<h3>Downloading OpenLane's dependencies…</a>"))
try:
    subprocess.check_call(
        ["nix", "profile", "install", ".#colab-env", "--accept-flake-config"],
        cwd=openlane_ipynb_path,
    )
except subprocess.CalledProcessError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to install binary dependencies using Nix…</h3>'
        )
    )

display(IPython.display.HTML("<h3>Downloading Python dependencies using PIP…</a>"))
try:
    subprocess.check_call(
        ["pip3", "install", "."],
        cwd=openlane_ipynb_path,
    )
except subprocess.CalledProcessError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to install Python dependencies using PIP…</h3>'
        )
    )
    raise e from None

display(IPython.display.HTML("<h3>Downloading PDK…</a>"))
import volare

volare.enable(
    volare.get_volare_home(pdk_root),
    pdk,
    open(
        os.path.join(openlane_ipynb_path, "openlane", "open_pdks_rev"),
        encoding="utf8",
    )
    .read()
    .strip(),
)

sys.path.insert(0, openlane_ipynb_path)
display(IPython.display.HTML("<h3>⭕️ Done.</a>"))

import logging

# Remove the stupid default colab logging handler
logging.getLogger().handlers.clear()

In [ ]:
import openlane

print(openlane.__version__)

### Creating the design

Now that OpenLane is set up, we can write a Verilog file as follows:

In [ ]:
%%writefile neuron.v
module neuron #(
    parameter Q          = 24,
    parameter N_INPUTS   = 4,
    parameter DATA_WIDTH = 32
) (
    input  wire                         clk,
    input  wire                         rst_n,
    input  wire                         af_sel,     // 0: tanh, 1: sigmoid
    input  wire signed [N_INPUTS*DATA_WIDTH-1:0] x_packed,
    input  wire signed [N_INPUTS*DATA_WIDTH-1:0] w_packed,
    input  wire signed [DATA_WIDTH-1:0] bias,
    output reg  signed [DATA_WIDTH-1:0] out
);

// Unpack Inputs/Weights
wire signed [DATA_WIDTH-1:0] x [N_INPUTS-1:0];
wire signed [DATA_WIDTH-1:0] w [N_INPUTS-1:0];

genvar p;
generate
    for (p=0; p<N_INPUTS; p=p+1) begin : UNPACK
        assign x[p] = x_packed[p*DATA_WIDTH +: DATA_WIDTH];
        assign w[p] = w_packed[p*DATA_WIDTH +: DATA_WIDTH];
    end
endgenerate

// Pipeline Stage 1: Input Buffering
reg signed [DATA_WIDTH-1:0] x_buf [N_INPUTS-1:0];
reg signed [DATA_WIDTH-1:0] w_buf [N_INPUTS-1:0];
reg signed [DATA_WIDTH-1:0] bias_buf;

integer j;

always @(posedge clk or negedge rst_n) begin
    if (!rst_n) begin
        for (j=0; j<N_INPUTS; j=j+1) begin
            x_buf[j] <= 0;
            w_buf[j] <= 0;
        end
        bias_buf <= 0;
    end else begin
        for (j=0; j<N_INPUTS; j=j+1) begin
            x_buf[j] <= x[j];
            w_buf[j] <= w[j];
        end
        bias_buf <= bias;
    end
end

// Pipeline Stage 2: Parallel Multiplication
wire [2*DATA_WIDTH-1:0] mult_results [N_INPUTS-1:0];
reg [2*DATA_WIDTH-1:0] mult_results_reg [N_INPUTS-1:0];

genvar i;

generate
    for (i=0; i<N_INPUTS; i=i+1) begin : MULT
        qmult #(.Q(Q), .N(DATA_WIDTH)) mult (
            .i_multiplicand(x_buf[i]),
            .i_multiplier(w_buf[i]),
            .o_result(mult_results[i]),
            .ovr()  // Overflow monitoring optional
        );

        always @(posedge clk or negedge rst_n) begin
            if (!rst_n) mult_results_reg[i] <= 0;
            else mult_results_reg[i] <= mult_results[i];
        end
    end
endgenerate

// Stage 3: Accumulation Tree
wire [2*DATA_WIDTH-1:0] sum_stage1 [N_INPUTS/2-1:0];
wire [2*DATA_WIDTH-1:0] sum_stage2;

generate
    // First level of addition (pairwise)
    for (i=0; i<N_INPUTS/2; i=i+1) begin : ADD_STAGE1
        qadd #(.Q(2*Q), .N(2*DATA_WIDTH)) adder (
            .a(mult_results_reg[2*i]),
            .b(mult_results_reg[2*i+1]),
            .c(sum_stage1[i]),
            .ovr()
        );
    end

    // Second level of addition (final sum)
    if (N_INPUTS > 2) begin : ADD_STAGE2
        qadd #(.Q(2*Q), .N(2*DATA_WIDTH)) final_adder (
            .a(sum_stage1[0]),
            .b(sum_stage1[1]),
            .c(sum_stage2),
            .ovr()
        );
    end else begin
        assign sum_stage2 = sum_stage1[0];
    end
endgenerate

// Stage 4: Bias Addition
wire [2*DATA_WIDTH-1:0] bias_ext = {bias_buf, {Q{1'b0}}}; // Q24 -> Q48
wire [2*DATA_WIDTH-1:0] sum_with_bias_wire;

qadd #(.Q(2*Q), .N(2*DATA_WIDTH)) bias_adder (
    .a(sum_stage2),
    .b(bias_ext),
    .c(sum_with_bias_wire),
    .ovr()
);

reg [DATA_WIDTH-1:0] sum_with_bias;
always @(posedge clk or negedge rst_n) begin
    if (!rst_n) sum_with_bias <= 0;
    else begin
        // Truncate from Q48 to Q24 with rounding
        sum_with_bias <= sum_with_bias_wire[Q +: DATA_WIDTH];
    end
end

// Pipeline Stage 4: Activation Function
wire signed [DATA_WIDTH-1:0] tanh_out, sigmoid_out;

tanh #(.Q(Q), .N(DATA_WIDTH)) tanh_inst (
    .x(sum_with_bias),
    .y(tanh_out)
);

sigmoid #(.Q(Q), .N(DATA_WIDTH)) sigmoid_inst (
    .x(sum_with_bias),
    .y(sigmoid_out)
);

// Activation function selection
reg signed [DATA_WIDTH-1:0] af_result;
always @(posedge clk or negedge rst_n) begin
    if (!rst_n) af_result <= 0;
    else af_result <= af_sel ? sigmoid_out : tanh_out;
end

// Pipeline Stage 5: Output Buffering
always @(posedge clk or negedge rst_n) begin
    if (!rst_n) out <= 0;
    else out <= af_result;
end

endmodule


module qadd #(
	//Parameterized values
	parameter Q = 15,
	parameter N = 32
	)
	(
    input [N-1:0] a,
    input [N-1:0] b,
    output [N-1:0] c,
    output ovr
    );
        reg [N-1:0] res;
        reg ovr_reg;
        reg [N-1:0] sum_mag;

        assign ovr = ovr_reg;
        assign c = res;

        always @(a,b) begin
        sum_mag = 0;
        ovr_reg = 0;
        res = 0;
        // both negative or both positive
        if(a[N-1] == b[N-1]) begin
            sum_mag = a[N-2:0] + b[N-2:0];
            res[N-2:0] = a[N-2:0] + b[N-2:0];
            res[N-1] = a[N-1];
            ovr_reg = sum_mag[N-1];
            end
        //	one of them is negative...
        else if(a[N-1] == 0 && b[N-1] == 1) begin
            if( a[N-2:0] > b[N-2:0] ) begin
                res[N-2:0] = a[N-2:0] - b[N-2:0];
                res[N-1] = 0;
                end
            else begin
                res[N-2:0] = b[N-2:0] - a[N-2:0];
                if (res[N-2:0] == 0)
                    res[N-1] = 0;
                else
                    res[N-1] = 1;
                end
            ovr_reg = 0;
            end
        else begin
            if( a[N-2:0] > b[N-2:0] ) begin
                res[N-2:0] = a[N-2:0] - b[N-2:0];
                if (res[N-2:0] == 0)
                    res[N-1] = 0;
                else
                    res[N-1] = 1;
                end
            else begin
                res[N-2:0] = b[N-2:0] - a[N-2:0];
                res[N-1] = 0;
                end
            ovr_reg = 0;
            end
        end
    endmodule


module qmult #(
    parameter Q = 24,
    parameter N = 32
    ) (
        input [N-1:0] i_multiplicand,
        input [N-1:0] i_multiplier,
        output [2*N-1:0] o_result,
        output reg ovr
    );

        reg [2*N-2:0] magnitude_product;
        reg sign_bit;
        // Threshold for overflow: (2^(N-1) - 1) << Q
        localparam THRESHOLD = ( (1 << (N-1)) - 1 ) << Q;

        always @(*) begin
            magnitude_product = i_multiplicand[N-2:0] * i_multiplier[N-2:0];
            sign_bit = i_multiplicand[N-1] ^ i_multiplier[N-1];
            // Overflow occurs if the product exceeds the maximum N-1 bit magnitude
            ovr = (magnitude_product > THRESHOLD);
        end

        assign o_result = {sign_bit, magnitude_product};

    endmodule


module qmac #(
    parameter Q = 5,
    parameter N = 8
    ) (
        input clk,
        input reset,
        input [N-1:0] a,
        input [N-1:0] b,
        output [2*N-1:0] result,
        output overflow
    );

        reg [2*N-1:0] accumulator;

        wire [2*N-1:0] product;
        wire mult_overflow;

        wire [2*N-1:0] sum;
        wire add_overflow;

        qmult #(.Q(Q), .N(N)) multiplier (
            .i_multiplicand(a),
            .i_multiplier(b),
            .o_result(product),
            .ovr(mult_overflow)
        );

        qadd #(.Q(2*Q), .N(2*N)) adder (
            .a(accumulator),
            .b(product),
            .c(sum),
            .ovr(add_overflow)
        );

        always @(posedge clk or posedge reset) begin
            if (reset)
                accumulator <= 0;
            else
                accumulator <= sum;
        end

        assign result = accumulator;
        assign overflow = add_overflow;

    endmodule


module sigmoid #(
    parameter Q = 24,
    parameter N = 32
    ) (
    input [N-1:0] x,
    output [N-1:0] y
    );
    // Thresholds
    localparam [N-1:0] T_1       = 32'b00000001000000000000000000000000; // 1.0
    localparam [N-1:0] T_2       = 32'b00000010000000000000000000000000; // 2.0
    localparam [N-1:0] T_3       = 32'b00000011000000000000000000000000; // 3.0
    localparam [N-1:0] T_4_5     = 32'b00000100100000000000000000000000; // 4.5
    localparam [N-1:0] T_8       = 32'b00001000000000000000000000000000; // 8.0

    // Coefficients for the piecewise linear approximation
    localparam signed [N-1:0] SLOPE1    = 32'b00000000000000001010010100100111; // 0.00252
    localparam signed [N-1:0] INTERCEPT1= 32'b00000000000001001100110011001101; // 0.01875
    localparam signed [N-1:0] SLOPE2    = 32'b00000000000001100000111100111101; // 0.02367
    localparam signed [N-1:0] INTERCEPT2= 32'b00000000000111010010110100100011; // 0.11397
    localparam signed [N-1:0] SLOPE3    = 32'b00000000000100011101101100100011; // 0.06975
    localparam signed [N-1:0] INTERCEPT3= 32'b00000000010000001000111110000110; // 0.25219
    localparam signed [N-1:0] SLOPE4    = 32'b00000000001001011111111000110011; // 0.14841
    localparam signed [N-1:0] INTERCEPT4= 32'b00000000011010001101010110100110; // 0.40951
    localparam signed [N-1:0] SLOPE5    = 32'b00000000001111010010100010001101; // 0.2389
    localparam signed [N-1:0] INTERCEPT5= 32'b00000000100000000000000000000000; // 0.5
    localparam signed [N-1:0] SLOPE6    = 32'b00000000001001011111111000110011; // 0.14841
    localparam signed [N-1:0] INTERCEPT6= 32'b00000000100101110010101001011010; // 0.59049
    localparam signed [N-1:0] SLOPE7    = 32'b00000000000100011101101100100011; // 0.06975
    localparam signed [N-1:0] INTERCEPT7= 32'b00000000101111110111000001111010; // 0.74781
    localparam signed [N-1:0] SLOPE8    = 32'b00000000000001100000111100111101; // 0.02367
    localparam signed [N-1:0] INTERCEPT8= 32'b00000000111000101101001011011101; // 0.88603
    localparam signed [N-1:0] SLOPE9    = 32'b00000000000000001010010100100111; // 0.00252
    localparam signed [N-1:0] INTERCEPT9= 32'b00000000111110110011001100110011; // 0.98125

    reg [N-1:0] slope;
    reg [N-1:0] intercept;
    wire [2*N-1:0] product;
    wire [N-1:0] product_trunc;
    wire [N-1:0] sum;
    wire ovr_mult, ovr_add;

    // magnitude and the sign bit of x
    wire [N-2:0] x_mag = x[N-2:0];
    wire x_sign = x[N-1];

    // coefficient selection logic
    always @(*) begin
        if (x_sign == 1'b1) begin  // Negative x
            if (x_mag >= T_8[N-2:0]) begin  // |x| >= 8.0
                slope = 32'b00000000000000000000000000000000;  // 0
                intercept = 32'b00000000000000000000000000000000;  // 0
            end else if (x_mag >= T_4_5[N-2:0] && x_mag < T_8[N-2:0]) begin  // 4.5 <= |x| < 8.0
                slope = SLOPE1;
                intercept = INTERCEPT1;
            end else if (x_mag >= T_3[N-2:0] && x_mag < T_4_5[N-2:0]) begin  // 3.0 <= |x| < 4.5
                slope = SLOPE2;
                intercept = INTERCEPT2;
            end else if (x_mag >= T_2[N-2:0] && x_mag < T_3[N-2:0]) begin  // 2.0 <= |x| < 3.0
                slope = SLOPE3;
                intercept = INTERCEPT3;
            end else if (x_mag >= T_1[N-2:0] && x_mag < T_2[N-2:0]) begin  // 1.0 <= |x| < 2.0
                slope = SLOPE4;
                intercept = INTERCEPT4;
            end else begin  // 0 < |x| < 1.0
                slope = SLOPE5;
                intercept = INTERCEPT5;
            end
        end else begin  // Positive x
            if (x_mag >= T_8[N-2:0]) begin  // x >= 8.0
                slope = 32'b00000000000000000000000000000000;  // 0
                intercept = 32'b00000001000000000000000000000000;  // 1.0
            end else if (x_mag >= T_4_5[N-2:0] && x_mag < T_8[N-2:0]) begin  // 4.5 <= x < 8.0
                slope = SLOPE9;
                intercept = INTERCEPT9;
            end else if (x_mag >= T_3[N-2:0] && x_mag < T_4_5[N-2:0]) begin  // 3.0 <= x < 4.5
                slope = SLOPE8;
                intercept = INTERCEPT8;
            end else if (x_mag >= T_2[N-2:0] && x_mag < T_3[N-2:0]) begin  // 2.0 <= x < 3.0
                slope = SLOPE7;
                intercept = INTERCEPT7;
            end else if (x_mag >= T_1[N-2:0] && x_mag < T_2[N-2:0]) begin  // 1.0 <= x < 2.0
                slope = SLOPE6;
                intercept = INTERCEPT6;
            end else begin  // 0 <= x < 1.0
                slope = SLOPE5;
                intercept = INTERCEPT5;
            end
        end
    end

    // Fixed-point multiplication
    qmult #(.Q(Q), .N(N)) mult (
        .i_multiplicand(x),
        .i_multiplier(slope),
        .o_result(product),
        .ovr(ovr_mult)
    );

    // Extract the proper bits from the product
    // For Q24, we need bits [55:24] of the 64-bit product
    assign product_trunc = {product[2*N-1], product[Q+N-2:Q]};

    // Fixed-point addition
    qadd #(.Q(Q), .N(N)) add (
        .a(product_trunc),
        .b(intercept),
        .c(sum),
        .ovr(ovr_add)
    );

    // Final output selection with proper sign handling
    assign y = (x_sign == 1'b1 && x_mag >= T_8[N-2:0]) ? 32'b00000000000000000000000000000000 :  // 0.0 for x <= -8.0
               (x_sign == 1'b0 && x_mag >= T_8[N-2:0]) ? 32'b00000001000000000000000000000000 :  // 1.0 for x >= 8.0
               sum;
endmodule


module tanh #(
    parameter Q = 24,
    parameter N = 32
    ) (
    input [N-1:0] x,
    output [N-1:0] y
    );
    localparam [N-1:0] TWO       = 32'b00000010000000000000000000000000; // +2.0
    localparam [N-1:0] ONE       = 32'b00000001000000000000000000000000; // +1.0
    localparam [N-1:0] NEG_ONE   = 32'b10000001000000000000000000000000; // -1.0

    wire [2*N-1:0] doubled_x;
    wire [N-1:0] sigmoid_result;
    wire [2*N-1:0] scaled_sigmoid;
    wire [N-1:0] final_result;

    qmult #(.Q(Q), .N(N)) double_mult (
        .i_multiplicand(x),
        .i_multiplier(TWO),
        .o_result(doubled_x),
        .ovr()
    );

    sigmoid #(.Q(Q), .N(N)) sigmoid_inst (
        .x(doubled_x[N-1:0]),
        .y(sigmoid_result)
    );

    qmult #(.Q(Q), .N(N)) scale_mult (
        .i_multiplicand(sigmoid_result),
        .i_multiplier(TWO),
        .o_result(scaled_sigmoid),
        .ovr()
    );

    qadd #(.Q(Q), .N(N)) subtr (
        .a(scaled_sigmoid[N-1:0]),
        .b(ONE),
        .c(final_result),
        .ovr()
    );

    assign y = final_result[N-1] ?
        {1'b1, final_result[N-2:0]} :  // Maintain sign-magnitude
        final_result;

endmodule

### Setting up the configuration

OpenLane requries to configure any Flow before using it. This is done using
the `config` module.

You can find the documentation for `Config.interactive` [here](https://openlane2.readthedocs.io/en/latest/reference/api/config/index.html#openlane.config.Config.interactive).



In [ ]:
from openlane.config import Config

Config.interactive(
    "neuron",
    PDK="sky130A",
    CLOCK_PORT="clk",
    CLOCK_NET="clk",
    CLOCK_PERIOD=10,
    PRIMARY_GDSII_STREAMOUT_TOOL="klayout",
)

### Running implementation steps

There are two ways to obtain OpenLane's built-in implementation steps:

* via directly importing from the `steps` module using its category:
    * `from openlane.steps import Yosys` then `Synthesis = Yosys.Synthesis`
* by using the step's id from the registry:
    * `from openlane.steps import Step` then `Synthesis = Step.factory.get("Yosys.Synthesis")`

You can find a full list of included steps here: https://openlane2.readthedocs.io/en/latest/reference/step_config_vars.html

In [ ]:
from openlane.steps import Step

* First, get the step (and display its help)...

In [ ]:
Synthesis = Step.factory.get("Yosys.Synthesis")

Synthesis.display_help()

* Then run it. Note you can pass step-specific configs using Python keyword
  arguments.

### Synthesis

We need to start by converting our high-level Verilog to one that just shows
the connections between small silicon patterns called "standard cells" in process
called Synthesis. We can do this by passing the Verilog files as a configuration
variable to `Yosys.Synthesis` as follows, then running it.

As this is the first step, we need to create an empty state and pass it to it.

In [ ]:
from openlane.state import State

synthesis = Synthesis(
    VERILOG_FILES=["./neuron.v"],
    state_in=State(),
)
synthesis.start()

In [ ]:
display(synthesis)

### Floorplanning

Floorplanning does two things:

* Determines the dimensions of the final chip.
* Creates the "cell placement grid" which placed cells must be aligned to.
    * Each cell in the grid is called a "site." Cells can occupy multiple
      sites, with the overwhelming majority of cells occupying multiple sites
      by width, and some standard cell libraries supporting varying heights as well.


In [ ]:
Floorplan = Step.factory.get("OpenROAD.Floorplan")

floorplan = Floorplan(
    state_in=synthesis.state_out
)
floorplan.start()

In [ ]:
display(floorplan)

### Tap/Endcap Cell Insertion

This places two kinds of cells on the floorplan:

* End cap/boundary cells: Added at the beginning and end of each row. True to
  their name, they "cap off" the core area of a design.
* Tap cells: Placed in a polka dot-ish fashion across the rows. Tap cells
  connect VDD to the nwell and the psubstrate to VSS, which the majority of cells
  do not do themselves to save area- but if you go long enough without one such
  connection you end up with the cell "latching-up"; i.e.; refusing to switch
  back to LO from HI.

  There is a maximum distance between tap cells enforced as part of every
  foundry process.

In [ ]:
TapEndcapInsertion = Step.factory.get("OpenROAD.TapEndcapInsertion")

tdi = TapEndcapInsertion(state_in=floorplan.state_out)
tdi.start()

In [ ]:
display(tdi)

### I/O Placement

This places metal pins at the edges of the design corresponding to the top level
inputs and outputs for your design. These pins act as the interface with other
designs when you integrate it with other designs.

In [ ]:
IOPlacement = Step.factory.get("OpenROAD.IOPlacement")

ioplace = IOPlacement(state_in=tdi.state_out)
ioplace.start()

In [ ]:
display(ioplace)

### Generating the Power Distribution Network (PDN)

This creates the power distribution network for your design, which is essentially
a plaid pattern of horizontal and vertical "straps" across the design that is
then connected to the rails' VDD and VSS (via the tap cells.)

You can find an explanation of how the power distribution network works at this
link: https://openlane2.readthedocs.io/en/latest/usage/hardening_macros.html#pdn-generation

While we typically don't need to mess with the PDN too much, the SPM is a small
design, so we're going to need to make the plaid pattern formed by the PDN a bit
smaller.

In [ ]:
GeneratePDN = Step.factory.get("OpenROAD.GeneratePDN")

pdn = GeneratePDN(
    state_in=ioplace.state_out,
    FP_PDN_VWIDTH=3,
    FP_PDN_HWIDTH=3,
    FP_PDN_VPITCH=40,
    FP_PDN_HPITCH=40,
)
pdn.start()

In [ ]:
display(pdn)

### Global Placement

Global Placement is deciding on a fuzzy, non-final location for each of the cells,
with the aim of minimizing the distance between cells that are connected
together (more specifically, the total length of the not-yet-created wires that
will connect them).

As you will see in the `.display()` in the second cell below, the placement is
considered "illegal", i.e., not properly aligned with the cell placement grid.
This is addressed by "Detailed Placement", also referred to as "placement
legalization", which is the next step.

In [ ]:
GlobalPlacement = Step.factory.get("OpenROAD.GlobalPlacement")

gpl = GlobalPlacement(state_in=pdn.state_out)
gpl.start()

In [ ]:
display(gpl)

### Detailed Placement

This aligns the fuzzy placement from before with the grid, "legalizing" it.

In [ ]:
DetailedPlacement = Step.factory.get("OpenROAD.DetailedPlacement")

dpl = DetailedPlacement(state_in=gpl.state_out)
dpl.start()

In [ ]:
display(dpl)

### Clock Tree Synthesis (CTS)

With the cells now having a final placement, we can go ahead and create what
is known as the clock tree, i.e., the hierarchical set of buffers used
for clock signal to minimize what is known as "clock skew"- variable delay
of the clock cycle from register to register because of factors such as metal
wire length, clock load (number of gates connected to the same clock buffer,)
et cetera.

The CTS step creates the cells and places the between the gaps in the detailed
placement above.

In [ ]:
CTS = Step.factory.get("OpenROAD.CTS")

cts = CTS(state_in=dpl.state_out)
cts.start()

In [ ]:
display(cts)

### Global Routing

Global routing "plans" the routes the wires between two gates (or gates and
I/O pins/the PDN) will take. The results of global routing (which are called
"routing guides") are stored in internal data structures and have no effect on
the actual design, so there is no `display()` statement.

In [ ]:
GlobalRouting = Step.factory.get("OpenROAD.GlobalRouting")

grt = GlobalRouting(state_in=cts.state_out)
grt.start()

### Detailed Routing

Detailed routing uses the guides from Global Routing to actually create wires
on the metal layers and connect the gates, making the connections finally physical.

This is typically the longest step in the flow.

In [ ]:
DetailedRouting = Step.factory.get("OpenROAD.DetailedRouting")

drt = DetailedRouting(state_in=grt.state_out)
drt.start()

In [ ]:
display(drt)

### Fill Insertion

Finally, as we're done placing all the essential cells, the only thing left to
do is fill in the gaps.

We prioritize the use of decap (decoupling capacitor) cells, which
further supports the power distribution network, but when there aren't any
small enough cells, we just use regular fill cells.

In [ ]:
FillInsertion = Step.factory.get("OpenROAD.FillInsertion")

fill = FillInsertion(state_in=drt.state_out)
fill.start()

In [ ]:
display(fill)

### Parasitics Extraction a.k.a. Resistance/Capacitance Extraction (RCX)

This step does not alter the design- rather, it computes the
[Parasitic elements](https://en.wikipedia.org/wiki/Parasitic_element_(electrical_networks))
of the circuit, which have an effect of timing, as we prepare to do the final
timing analysis.

The parasitic elements are saved in the **Standard Parasitics Exchange Format**,
or SPEF. OpenLane creates a SPEF file for each interconnect corner as described in
the [Corners and STA](https://openlane2.readthedocs.io/en/latest/usage/corners_and_sta.html)
section of the documentation.

In [ ]:
RCX = Step.factory.get("OpenROAD.RCX")

rcx = RCX(state_in=fill.state_out)
rcx.start()

### Static Timing Analysis (Post-PnR)

STA is a process that verifies that a chip meets certain constraints on clock
and data timings to run at its rated clock speed. See [Corners and STA](https://openlane2.readthedocs.io/en/latest/usage/corners_and_sta.html)
in the documentation for more info.

---

This step generates two kinds of files:
* `.lib`: Liberty™-compatible Library files. Can be used to do static timing
  analysis when creating a design with this design as a sub-macro.
* `.sdf`: Standard Delay Format. Can be used with certain simulation software
  to do *dynamic* timing analysis.

Unfortunately, the `.lib` files coming out of OpenLane right now are not super
reliable for timing purposes and are only provided for completeness.
When using OpenLane-created macros withing other designs, it is best to use the
macro's final netlist and extracted parasitics instead.

In [ ]:
STAPostPNR = Step.factory.get("OpenROAD.STAPostPNR")

sta_post_pnr = STAPostPNR(state_in=rcx.state_out,STA_WRITE_LIB=True)
sta_post_pnr.start()

### Stream-out

Stream-out is the process of converting the designs from the abstract formats
using during floorplanning, placement and routing into a concrete format called
GDSII (lit. Graphic Design System 2), which is the final file that is then sent
for fabrication.

In [ ]:
StreamOut = Step.factory.get("KLayout.StreamOut")

gds = StreamOut(state_in=sta_post_pnr.state_out)
gds.start()

In [ ]:
display(gds)

### Design Rule Checks (DRC)

DRC determines that the final layout does not violate any of the rules set by
the foundry to ensure the design is actually manufacturable- for example,
not enough space between two wires, *too much* space between tap cells, and so
on.

A design not passing DRC will typically be rejected by the foundry, who
also run DRC on their side.

In [ ]:
DRC = Step.factory.get("Magic.DRC")

drc = DRC(state_in=gds.state_out)
drc.start()

### SPICE Extraction for Layout vs. Schematic Check

This step tries to reconstruct a SPICE netlist from the GDSII file, so it can
later be used for the **Layout vs. Schematic** (LVS) check.

In [ ]:
SpiceExtraction = Step.factory.get("Magic.SpiceExtraction")

spx = SpiceExtraction(state_in=drc.state_out)
spx.start()

### Layout vs. Schematic (LVS)

A comparison between the final Verilog netlist (from PnR) and the final
SPICE netlist (extracted.)

This check effectively compares the physically implemented circuit to the final
Verilog netlist output by OpenROAD.

The idea is, if there are any disconnects, shorts or other mismatches in the
physical implementation that do not exist in the logical view of the design,
they would be caught at this step.

Common issues that result in LVS violations include:
* Lack of fill cells or tap cells in the design
* Two unrelated signals to be shorted, or a wire to be disconnected (most
  commonly seen with misconfigured PDN)

Chips with LVS errors are typically dead on arrival.

In [ ]:
LVS = Step.factory.get("Netgen.LVS")

lvs = LVS(state_in=spx.state_out)
lvs.start()